# DealScout - Phase 1, Step 3: Evaluation, Baselines & Traditional ML

Evaluate baselines and traditional ML models, scored with the shared `evaluate` harness

In [ ]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

from dealscout.core.evaluator import evaluate
from dealscout.core.items import Item

In [ ]:
LITE_MODE = True

In [ ]:
username = "Ankit-Singh-ai"
dataset = f"{username}/dealscout_items_lite" if LITE_MODE else f"{username}/dealscout_items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [14]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }
    
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

## Traditional Linear Regression!

In [15]:
np.random.seed(42)

# Separate features and target
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Train a Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# Predict the test set and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

weight: 3.5687444142553044
weight_unknown: 20.90175988023751
text_length: 0.20343123739152702
Intercept: 40.91238780791184
Mean Squared Error: 20096.925335647466
R-squared Score: 0.1609102308229894


In [16]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [17]:
evaluate(linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$79 $51 $62 $66 $71 $50 $9 $50 $82 $191 $574 $233 $113 $58 $28 $97 $50 $62 $39 $14 $18 $14 $75 $11 $193 $318 $359 $91 $35 $36 $104 $78 $27 $16 $79 $676 $65 $66 $70 $67 $64 $36 $61 $66 $104 $93 $104 $83 $25 $60 $84 $5 $347 $2 $79 $20 $110 $83 $23 $123 $111 $49 $19 $3 $241 $17 $82 $261 $6 $68 $85 $105 $155 $89 $91 $80 $3 $98 $99 $88 $72 $86 $86 $8 $84 $43 $81 $176 $62 $60 $88 $42 $80 $71 $91 $117 $78 $8 $146 $418 $33 $7 $106 $21 $122 $1 $83 $298 $104 $198 $62 $154 $106 $59 $71 $53 $103 $94 $82 $14 $99 $43 $51 $51 $127 $47 $86 $102 $72 $16 $53 $9 $47 $72 $45 $84 $81 $40 $9 $11 $39 $132 $55 $85 $3 $71 $71 $364 $5 $72 $93 $26 $53 $21 $90 $119 $103 $40 $43 $79 $160 $92 $67 $116 $725 $100 $279 $74 $84 $101 $86 $100 $244 $78 $139 $58 $95 $7 $50 $38 $289 $117 $38 $94 $75 $96 $6 $77 $57 $79 $79 $78 $108 $21 $14 $97 $50 $109 $90 $114 

In [18]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)

In [19]:
# Here are the 1,000 most common words that it picked, not including "stop words":

selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

Number of selected words: 2000
Selected words: ['items' 'jack' 'jacket' 'jeep' 'jigsaw' 'joint' 'joints' 'keeping'
 'keeps' 'key' 'keyboard' 'keypad' 'keys' 'kg' 'khz' 'kia' 'kickstand'
 'kids' 'king' 'kingston']


In [20]:
regressor = LinearRegression()
regressor.fit(X, prices)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2000,)","[ 21.82, 36.58, -3.82,...,-14.15,-10.18, 84.38]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-26.06
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2000


In [21]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [22]:
evaluate(natural_language_linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$86 $120 $55 $70 $191 $230 $24 $41 $8 $65 $480 $188 $61 $132 $19 $139 $64 $31 $25 $27 $39 $86 $51 $20 $318 $360 $149 $36 $38 $32 $75 $58 $3 $26 $120 $338 $10 $106 $142 $73 $151 $110 $88 $9 $89 $42 $74 $72 $14 $35 $41 $49 $154 $42 $78 $167 $111 $194 $15 $12 $40 $3 $55 $21 $426 $35 $26 $191 $11 $224 $24 $46 $92 $111 $19 $49 $140 $11 $20 $125 $7 $37 $71 $43 $61 $103 $5 $149 $52 $255 $30 $142 $84 $41 $52 $133 $64 $56 $213 $108 $88 $57 $37 $35 $3 $51 $14 $162 $31 $71 $15 $79 $206 $13 $33 $123 $119 $31 $49 $19 $16 $159 $8 $27 $65 $24 $20 $45 $58 $59 $5 $63 $181 $14 $26 $47 $1 $151 $99 $104 $49 $141 $5 $203 $139 $74 $19 $317 $19 $22 $19 $263 $30 $37 $43 $56 $163 $16 $102 $13 $63 $26 $42 $13 $387 $9 $85 $50 $2 $10 $10 $61 $318 $65 $69 $82 $8 $74 $0 $120 $386 $11 $58 $21 $107 $39 $5 $67 $150 $27 $38 $52 $33 $36 $32 $59 $45 $28 $15 $36 

## Random Forest model

The Random Forest is a type of "**ensemble**" algorithm, meaning that it combines many smaller algorithms to make better predictions.

It uses a very simple kind of machine learning algorithm called a **decision tree**. A decision tree makes predictions by examining the values of features in the input. Like a flow chart with IF statements. Decision trees are very quick and simple, but they tend to overfit.

In our case, the "features" are the elements of the Vector - in other words, it's the number of times that a particular word appears in the product description.

So you can think of it something like this:

**Decision Tree**  
\- IF the word "TV" appears more than 3 times THEN  
-- IF the word "LED" appears more than 2 times THEN  
--- IF the word "HD" appears at least once THEN  
---- Price = $500


With Random Forest, multiple decision trees are created. Each one is trained with a different random subset of the data, and a different random subset of the features. You can see above that we specify 100 trees, which is the default.

Then the Random Forest model simply takes the average of all its trees to product the final result.

In [23]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",4
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"ma

In [24]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [25]:
evaluate(random_forest, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$65 $69 $7 $17 $116 $146 $76 $63 $27 $269 $511 $287 $44 $86 $1 $7 $46 $109 $29 $55 $17 $25 $13 $30 $139 $313 $166 $119 $30 $50 $53 $80 $88 $20 $36 $361 $45 $29 $146 $93 $150 $47 $17 $52 $150 $32 $48 $41 $63 $11 $9 $30 $238 $30 $286 $45 $33 $154 $6 $50 $106 $5 $61 $6 $451 $6 $23 $205 $36 $125 $12 $25 $61 $134 $4 $85 $106 $17 $30 $81 $66 $72 $35 $50 $17 $32 $22 $105 $4 $114 $27 $173 $1 $84 $61 $63 $3 $40 $139 $286 $33 $5 $12 $70 $33 $18 $118 $192 $21 $185 $37 $67 $53 $31 $64 $40 $92 $10 $210 $26 $33 $59 $48 $37 $36 $27 $39 $46 $119 $8 $22 $44 $63 $14 $38 $59 $26 $6 $86 $30 $11 $154 $0 $100 $96 $68 $36 $320 $85 $4 $20 $101 $51 $73 $50 $90 $109 $39 $37 $15 $20 $2 $4 $31 $388 $46 $256 $28 $6 $35 $19 $7 $219 $30 $14 $58 $48 $7 $39 $9 $330 $15 $62 $84 $132 $57 $53 $77 $4 $39 $34 $54 $28 $14 $6 $22 $74 $28 $8 $17 

## Introducing XGBoost

Like Random Forest, XGBoost is also an ensemble model that combines multiple decision trees.

But unlike Random Forest, XGBoost builds one tree after another, with each next tree correcting for errors in the prior trees, using 'gradient descent'.

It's much faster than Random Forest, so we can run it for the full dataset, and it's typically better at generalizing.

In [27]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [28]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [29]:
evaluate(xg_boost, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$82 $81 $30 $16 $107 $160 $39 $3 $16 $21 $601 $245 $107 $107 $0 $17 $55 $13 $15 $15 $11 $25 $5 $68 $234 $321 $95 $31 $40 $28 $84 $55 $9 $9 $54 $124 $24 $69 $164 $68 $154 $88 $63 $8 $118 $113 $37 $57 $43 $8 $30 $37 $126 $8 $236 $95 $61 $193 $2 $41 $72 $8 $71 $35 $444 $33 $46 $208 $50 $100 $2 $43 $92 $106 $13 $200 $222 $10 $34 $117 $22 $95 $87 $40 $19 $50 $63 $124 $26 $226 $34 $196 $27 $3 $25 $93 $124 $26 $199 $168 $114 $40 $23 $61 $5 $91 $175 $207 $31 $66 $39 $64 $72 $54 $79 $140 $129 $29 $136 $42 $45 $98 $35 $33 $135 $4 $16 $97 $179 $99 $38 $52 $141 $35 $9 $70 $74 $48 $164 $93 $2 $148 $55 $89 $82 $96 $36 $260 $11 $5 $19 $163 $46 $60 $25 $96 $117 $16 $21 $4 $74 $21 $8 $13 $397 $2 $147 $50 $15 $11 $22 $26 $209 $32 $31 $47 $50 $32 $37 $58 $423 $12 $137 $30 $113 $59 $61 $45 $65 $30 $47 $20 $16 $17 $4 $71 $57 $41 $40 $26 